# **Caso: Análisis de la Copa Mundial Femenina**

## **Introducción**

La Copa Mundial Femenina de la FIFA ha experimentado un crecimiento exponencial desde su primera edición en 1991, atrayendo una audiencia global y consolidando el fútbol femenino como un pilar del deporte internacional. Con la evolución del torneo, ha surgido la necesidad de analizar tendencias en el desempeño de las selecciones, patrones de juego y factores que contribuyen al
éxito en la competición.

El fin de este estudio es realizar  un análisis detallado del rendimiento de los equipos en las distintas ediciones del torneo, identificando patrones en la cantidad de goles, frecuencia de victorias y desempeño a lo largo de los años. Se busca descubrir qué selecciones han dominado históricamente y cómo ha evolucionado el nivel de competitividad.

## **Importación de librerías necesarias**

In [ ]:
import pandas as pd
import plotly.express as px
from IPython.display import display, Markdown
import plotly.graph_objects as go
from plotly.subplots import make_subplots

## **Funciones utilizadas**

### **Funciones para realizar gráficos**

In [ ]:
def VisualizacionInicial(df, nombre):

    # Hacer una copia y convertir tipos de datos
    df_vis = df.copy()
    for col in df_vis.select_dtypes(include=['Int64']):
        df_vis[col] = df_vis[col].astype('int64')

    paleta = ["#4A148C", "#6A1B9A", "#8E24AA", "#AB47BC", "#CE93D8"]
    fuente = "Inter"
    tamaño_letra = 16
    tamaño_titulo = 18

    # Gráfico circular de tipos de datos
    tipos = df_vis.dtypes.value_counts().reset_index()
    tipos.columns = ['Tipo', 'Cantidad']
    tipos['Tipo'] = tipos['Tipo'].astype(str)

    fig_pie = px.pie(
        tipos,
        values='Cantidad',
        names='Tipo',
        title=f'<b style="font-size:{tamaño_titulo}px">Distribución de Tipos de Datos</b><br><sup>{nombre}</sup>',
        color_discrete_sequence=paleta,
        hole=0.4
    )

    fig_pie.update_traces(
        textposition='inside',
        textinfo='percent+label',
        marker=dict(line=dict(color='white', width=1)),
        textfont_size=tamaño_letra
    )

    fig_pie.update_layout(
        font_family=fuente,
        font_size=tamaño_letra,
        legend=dict(
            orientation="h",
            yanchor="bottom",
            y=1.02,
            font=dict(size=tamaño_letra)
        ),
        hoverlabel=dict(
            font_size=tamaño_letra,
            font_family=fuente
        ),
        paper_bgcolor='white',
        plot_bgcolor='white'
    )

    # Gráfico de valores nulos
    nulos = df_vis.isnull().mean().round(4)*100
    nulos = nulos[nulos > 0].reset_index()
    nulos.columns = ['Variable', 'Porcentaje']
    nulos['Porcentaje'] = nulos['Porcentaje'].astype('float64')

    if not nulos.empty:
        fig_nulos = px.bar(
            nulos.sort_values('Porcentaje', ascending=False),
            x='Variable',
            y='Porcentaje',
            title=f'<b style="font-size:{tamaño_titulo}px">Porcentaje de Valores Nulos</b><br><sup>{nombre}</sup>',
            text='Porcentaje',
            color='Porcentaje',
            color_continuous_scale=paleta
        )

        fig_nulos.update_traces(
            texttemplate='%{text:.2f}%',
            textposition='outside',
            marker_line_color='white',
            marker_line_width=1.5,
            textfont_size=tamaño_letra
        )

        fig_nulos.update_layout(
            yaxis_title='Porcentaje de nulos (%)',
            xaxis_title='',
            font_family=fuente,
            font_size=tamaño_letra,
            coloraxis_showscale=False,
            hovermode='x unified',
            paper_bgcolor='white',
            plot_bgcolor='white',
            xaxis=dict(tickfont=dict(size=tamaño_letra)),
            yaxis=dict(tickfont=dict(size=tamaño_letra))
        )

    # Gráfico temporal
    if 'Year' in df_vis.columns:
        temporal = df_vis['Year'].value_counts().sort_index().reset_index()
        temporal.columns = ['Year', 'Conteo']
        temporal['Year'] = temporal['Year'].astype('int64')

        fig_line = px.line(
            temporal,
            x='Year',
            y='Conteo',
            title=f'<b style="font-size:{tamaño_titulo}px">Distribución Temporal</b><br><sup>{nombre}</sup>',
            markers=True,
            color_discrete_sequence=[paleta[0]]
        )

        fig_line.update_traces(
            line_width=3,
            marker=dict(size=10, line=dict(width=2, color='white')),
            textfont_size=tamaño_letra
        )

        fig_line.update_layout(
            xaxis_title='Año',
            yaxis_title='Registros',
            font_family=fuente,
            font_size=tamaño_letra,
            hovermode='x unified',
            paper_bgcolor='white',
            plot_bgcolor='white',
            xaxis=dict(
                tickfont=dict(size=tamaño_letra),
                titlefont=dict(size=tamaño_letra)
            ),
            yaxis=dict(
                tickfont=dict(size=tamaño_letra),
                titlefont=dict(size=tamaño_letra)
            )
        )

    # Mostrar gráficos
    fig_pie.show()

    if not nulos.empty:
        fig_nulos.show()
    else:
        print("\n✅ No hay valores nulos para visualizar")

    if 'Year' in df_vis.columns:
        fig_line.show()
    else:
        print("\nℹ️ No se encontró columna 'Year' para análisis temporal")

In [ ]:
def GraficoAsistenciaPromedio(df):
    paleta = ["#4A148C", "#6A1B9A", "#8E24AA"]
    fuente = "Inter"
    tamaño_letra = 14
    tamaño_titulo = 16

    fig = make_subplots(rows=1, cols=2,
                       subplot_titles=(
                           "<b>Evolución de Asistencia Promedio</b>",
                           "<b>Relación: Asistencia vs Partidos</b>"
                       ))

    fig.add_trace(
        go.Scatter(
            x=df["Year"],
            y=df["AttendanceAvg"],
            mode="lines+markers",
            line=dict(color=paleta[0], width=3),
            marker=dict(
                size=10,
                color=paleta[0],
                line=dict(width=2, color='white')
            ),
            name="Asistencia promedio",
            hovertemplate="Año: %{x}<br>Asistencia: %{y:,.0f}<extra></extra>"
        ),
        row=1, col=1
    )

    fig.add_trace(
        go.Scatter(
            x=df["Matches"],
            y=df["AttendanceAvg"],
            mode="markers",
            marker=dict(
                size=12,
                color=df["Year"],
                colorscale=paleta,
                line=dict(width=1, color='white'),
                showscale=True,
                colorbar=dict(title="Año")
            ),
            text=df["Year"],
            hovertemplate="Partidos: %{x}<br>Asistencia: %{y:,.0f}<br>Año: %{text}<extra></extra>",
            name="Relación"
        ),
        row=1, col=2
    )

    fig.update_layout(
        font_family=fuente,
        font_size=tamaño_letra,
        plot_bgcolor='white',
        paper_bgcolor='white',
        hoverlabel=dict(
            font_size=tamaño_letra,
            font_family=fuente
        ),
        showlegend=False,
        margin=dict(l=50, r=50, t=80, b=50),
        height=500
    )

    fig.update_xaxes(
        title_text="Año",
        row=1, col=1,
        title_font=dict(size=tamaño_letra))

    fig.update_yaxes(
        title_text="Asistencia promedio",
        row=1, col=1,
        title_font=dict(size=tamaño_letra))

    fig.update_xaxes(
        title_text="Número de partidos",
        row=1, col=2,
        title_font=dict(size=tamaño_letra))

    fig.update_yaxes(
        title_text="Asistencia promedio",
        row=1, col=2,
        title_font=dict(size=tamaño_letra))

    fig.update_annotations(
        font_size=tamaño_titulo,
        font_family=fuente
    )

    fig.show()

In [ ]:
def GraficoPartidosEmpatados(df):

    paleta = ["#6A1B9A", "#8E24AA", "#AB47BC"]
    fuente = "Inter"
    tamaño_letra = 14
    tamaño_titulo = 16

    fig = make_subplots(rows=1, cols=2,
                       subplot_titles=(
                           "<b>Evolución de Partidos Empatados</b>",
                           "<b>Relación: Empates vs Tamaño del Torneo</b>"
                       ))

    fig.add_trace(
        go.Scatter(
            x=df["Year"],
            y=df["Draws"],
            mode="lines+markers",
            line=dict(color=paleta[0], width=3),
            marker=dict(
                size=10,
                color=paleta[0],
                line=dict(width=2, color='white')
            ),
            name="Empates",
            hovertemplate="Año: %{x}<br>Empates: %{y}<extra></extra>"
        ),
        row=1, col=1
    )

    fig.add_trace(
        go.Scatter(
            x=df["Teams"],
            y=df["Draws"],
            mode="markers",
            marker=dict(
                size=12,
                color=df["Year"],
                colorscale=paleta,
                line=dict(width=1, color='white'),
                showscale=True,
                colorbar=dict(title="Año")
            ),
            text=df["Year"],
            hovertemplate="Equipos: %{x}<br>Empates: %{y}<br>Año: %{text}<extra></extra>",
            name="Relación"
        ),
        row=1, col=2
    )

    fig.update_layout(
        font_family=fuente,
        font_size=tamaño_letra,
        plot_bgcolor='white',
        paper_bgcolor='white',
        hoverlabel=dict(
            font_size=tamaño_letra,
            font_family=fuente
        ),
        showlegend=False,
        margin=dict(l=50, r=50, t=80, b=50),
        height=500
    )

    fig.update_xaxes(title_text="Año", row=1, col=1, title_font_size=tamaño_letra)
    fig.update_yaxes(title_text="Partidos empatados", row=1, col=1, title_font_size=tamaño_letra)
    fig.update_xaxes(title_text="Número de equipos", row=1, col=2, title_font_size=tamaño_letra)
    fig.update_yaxes(title_text="Partidos empatados", row=1, col=2, title_font_size=tamaño_letra)

    fig.update_annotations(font_size=tamaño_titulo)

    fig.show()

In [ ]:
def GraficoPromedioGoles(df):
    paleta = ["#4A148C", "#5E35B1", "#7E57C2"]
    fuente = "Inter"
    tamaño_letra = 14
    tamaño_titulo = 16

    fig = make_subplots(rows=1, cols=2,
                       subplot_titles=(
                           "<b>Evolución del Promedio de Goles</b>",
                           "<b>Relación: Goles vs Asistencia</b>"
                       ),
                       horizontal_spacing=0.15)

    fig.add_trace(
        go.Scatter(
            x=df["Year"],
            y=df["Avg_Goals"],
            mode="lines+markers",
            line=dict(color=paleta[0], width=3),
            marker=dict(
                size=10,
                color=paleta[0],
                line=dict(width=2, color='white')
            ),
            name="Promedio goles",
            hovertemplate="<b>Año %{x}</b><br>Promedio: %{y:.2f} goles<extra></extra>"
        ),
        row=1, col=1
    )

    fig.add_trace(
        go.Scatter(
            x=df["AttendanceAvg"],
            y=df["Avg_Goals"],
            mode="markers",
            marker=dict(
                size=12,
                color=df["Year"],
                colorscale=paleta,
                line=dict(width=1, color='white'),
                showscale=True,
                colorbar=dict(title="Año", thickness=15)
            ),
            text=df[["Year", "Host"]].apply(lambda x: f"{x[0]} ({x[1]})", axis=1),
            hovertemplate="<b>%{text}</b><br>Asistencia: %{x:,.0f}<br>Goles: %{y:.2f}<extra></extra>",
            name="Relación"
        ),
        row=1, col=2
    )

    fig.update_layout(
        font_family=fuente,
        font_size=tamaño_letra,
        plot_bgcolor='white',
        paper_bgcolor='white',
        hoverlabel=dict(
            font_size=tamaño_letra,
            font_family=fuente,
            bgcolor="white",
            bordercolor="#E0E0E0"
        ),
        showlegend=False,
        margin=dict(l=50, r=50, t=90, b=60),
        height=500
    )

    fig.update_xaxes(
        title_text="Año",
        row=1, col=1,
        title_font=dict(size=tamaño_letra),
        tickfont=dict(size=tamaño_letra-2)
    )

    fig.update_yaxes(
        title_text="Promedio de goles",
        row=1, col=1,
        title_font=dict(size=tamaño_letra),
        tickfont=dict(size=tamaño_letra-2),
        range=[min(df["Avg_Goals"])*0.9, max(df["Avg_Goals"])*1.1]
    )

    fig.update_xaxes(
        title_text="Asistencia promedio",
        row=1, col=2,
        title_font=dict(size=tamaño_letra),
        tickfont=dict(size=tamaño_letra-2),
        tickformat=","
    )

    fig.update_yaxes(
        title_text="Promedio de goles",
        row=1, col=2,
        title_font=dict(size=tamaño_letra),
        tickfont=dict(size=tamaño_letra-2)
    )

    fig.update_annotations(
        font_size=tamaño_titulo,
        font_family=fuente,
        y=1.05
    )

    fig.show()

### **Otras funciones**

In [ ]:
def AnalisisEstructura(df, nombre):

    print(f"🔍 Análisis estructural del dataset: {nombre}")

    # Dimensiones del dataset
    print(f"\n📌 Dimensiones: {df.shape[0]} registros x {df.shape[1]} variables\n")

    # Duplicados
    duplicados = df.duplicated().sum()
    print(f"\n📌 Filas duplicadas: {duplicados} ({duplicados/len(df):.2%})\n")

    # Valores nulos
    nulos = pd.DataFrame({
        'Variable': df.isnull().sum().index,
        'Valores nulos': df.isnull().sum().values,
        '% Nulos': (df.isnull().mean()*100).round(2)
    }).query('`Valores nulos` > 0')

    if not nulos.empty:
        nulos_styled = (nulos.style
                       .set_caption("Variables con valores nulos")
                       .format({'% Nulos': "{:.2f}%"})
                       .background_gradient(subset=['Valores nulos', '% Nulos'], cmap='Purples'))
        display(nulos_styled)
    else:
        print("✅ No se encontraron valores nulos")

    print("")

    # Tipos de variables
    tipos = df.dtypes.reset_index()
    tipos.columns = ['Variable', 'Tipo']
    tipos_styled = (tipos.style
                   .set_caption("Tipos de variables")
                   .background_gradient(cmap='Purples_r')
                   .set_properties(**{'text-align': 'left'}))
    display(tipos_styled)

In [ ]:
def ValidacionCruzadaTablas(df_matches, df_women):
    print("🔎 Validación cruzada entre `matches` y `world_cup_women`\n")

    df_matches['Year'] = df_matches['Year'].astype(int)
    df_women['Year'] = df_women['Year'].astype(int)
    df_matches['Date'] = pd.to_datetime(df_matches['Date'], errors='coerce')

    # Años faltantes
    years_matches = set(df_matches['Year'].unique())
    years_women = set(df_women['Year'].unique())
    years_faltantes = sorted(years_matches - years_women)

    print("\n📌 Años en `matches` no presentes en `world_cup_women`:")
    if years_faltantes:
        print(f"- {', '.join(map(str, years_faltantes))}")
    else:
        print("- ✅ Todos los años de `matches` están presentes en `world_cup_women`.\n")

    # Validación del anfitrión
    anfitriones_matches = df_matches.groupby('Year')['Host'].agg(lambda x: x.mode().iloc[0]).reset_index(name='Host_matches')
    anfitriones_women = df_women[['Year', 'Host']].rename(columns={'Host': 'Host_women'})
    comparacion_host = anfitriones_matches.merge(anfitriones_women, on='Year', how='left')
    comparacion_host['Coincide'] = comparacion_host['Host_matches'] == comparacion_host['Host_women']
    print("\n📌 Comparación de país anfitrión por edición:")
    display(comparacion_host)

    # Comparación del número de partidos
    partidos_matches = df_matches.groupby('Year').size().reset_index(name='Partidos_matches')
    partidos_women = df_women[['Year', 'Matches']]
    comparacion_partidos = partidos_matches.merge(partidos_women, on='Year', how='left')
    comparacion_partidos['Diferencia'] = comparacion_partidos['Partidos_matches'] - comparacion_partidos['Matches']
    print("\n📌 Comparación del número de partidos por edición:")
    display(comparacion_partidos)

    # Comparación de audiencia
    audiencia_matches = df_matches.groupby('Year')['Attendance'].sum().reset_index(name='Attendance_matches')
    audiencia_women = df_women[['Year', 'Attendance']]
    comparacion_audiencia = audiencia_matches.merge(audiencia_women, on='Year', how='left')
    comparacion_audiencia['Diferencia'] = comparacion_audiencia['Attendance_matches'] - comparacion_audiencia['Attendance']
    print("\n📌 Comparación de la audiencia total por edición:")
    display(comparacion_audiencia)

    # Partidos en años no registrados
    partidos_fuera = df_matches[~df_matches['Year'].isin(df_women['Year'])]
    print(f"\n📌 Número de partidos en años no registrados en `df_women`: `{partidos_fuera.shape[0]}`")

    # Verificación Campeón y Subcampeón
    top2_verificado = []
    for year in df_women['Year'].dropna().unique():
        final_partido = df_matches[(df_matches['Year'] == year) & (df_matches['Round'].str.strip().str.lower() == "final")]

        if final_partido.empty:
            top2_verificado.append({'Year': year, 'Final_Check': False, 'Motivo': 'No se encontró partido con Round == Final'})
            continue

        home = final_partido.iloc[0]['home_team']
        away = final_partido.iloc[0]['away_team']
        equipos_final = set([home, away])

        campeon = df_women[df_women['Year'] == year]['Champion'].values[0]
        subcampeon = df_women[df_women['Year'] == year]['Runner-Up'].values[0]
        top2_set = set([campeon, subcampeon])

        coincide = equipos_final == top2_set
        motivo = '✔️ Coinciden' if coincide else f'❌ No coinciden: Final fue {equipos_final}, Top2 fue {top2_set}'

        top2_verificado.append({'Year': year, 'Final_Check': coincide, 'Motivo': motivo})

    top2_df = pd.DataFrame(top2_verificado)
    print("\n📌 Verificación de Top 2 (campeón y subcampeón):")
    display(top2_df)

In [ ]:
def ResumenPorEquipoPipeline(df_matches):
    return (

        # Se combinan el equipo local (home) y el visitante (away)
        pd.concat([
            df_matches[['Year', 'Host', 'home_team', 'home_score', 'away_score', 'Attendance']]
            .rename(columns={'home_team': 'Team', 'home_score': 'Goals_For', 'away_score': 'Goals_Against'}),

            df_matches[['Year', 'Host', 'away_team', 'away_score', 'home_score', 'Attendance']]
            .rename(columns={'away_team': 'Team', 'away_score': 'Goals_For', 'home_score': 'Goals_Against'})
        ])

        # Clasifica el resultado del partido
        .assign(
            Result=lambda df: df.apply(
                lambda row: 'Win' if row.Goals_For > row.Goals_Against
                else 'Loss' if row.Goals_For < row.Goals_Against
                else 'Draw', axis=1
            )
        )
        .groupby(['Year', 'Host', 'Team'], as_index=False)

        # Métricas por equipo
        .agg(
            Matches_Played=('Result', 'count'),
            Goals_Scored=('Goals_For', 'sum'),
            Goals_Received=('Goals_Against', 'sum'),
            Avg_Goals_Scored=('Goals_For', 'mean'),
            Avg_Goals_Received=('Goals_Against', 'mean'),
            Wins=('Result', lambda x: (x == 'Win').sum()),
            Draws=('Result', lambda x: (x == 'Draw').sum()),
            Losses=('Result', lambda x: (x == 'Loss').sum()),
            Avg_Attendance=('Attendance', 'mean')
        )
    )

In [ ]:
def EquiposPromediosExtremos(resumen_equipo):

    # Agrupación global por equipo
    resumen_global = (
        resumen_equipo
        .groupby('Team')
        .agg(
            Total_Goles_Anotados=('Goals_Scored', 'sum'),
            Total_Goles_Recibidos=('Goals_Received', 'sum'),
            Partidos=('Matches_Played', 'sum')
        )

        # Cálculo de promedios
        .assign(
            Promedio_Anotados=lambda df: df['Total_Goles_Anotados'] / df['Partidos'],
            Promedio_Recibidos=lambda df: df['Total_Goles_Recibidos'] / df['Partidos']
        )
        .reset_index()
    )

    #  Identificación de extremos
    mayor_anotador = resumen_global.loc[resumen_global['Promedio_Anotados'].idxmax()]
    mayor_recibidor = resumen_global.loc[resumen_global['Promedio_Recibidos'].idxmax()]

    return {
        "Mayor promedio anotador": mayor_anotador[['Team', 'Promedio_Anotados']],
        "Mayor promedio recibido": mayor_recibidor[['Team', 'Promedio_Recibidos']]
    }

In [ ]:

def CompararConMundialesParticipados(resumen_equipo, equipos_destacados):

    # Contar en cuántos años distintos aparece cada equipo
    participaciones = resumen_equipo.groupby('Team')['Year'].nunique().reset_index()
    participaciones.columns = ['Team', 'Mundiales_Jugados']

    # Nombres de equipos a comparar
    equipos_a_comparar = set()
    for info in equipos_destacados.values():
        equipos_a_comparar.add(info['Team'])

    resumen_top = resumen_equipo[resumen_equipo['Team'].isin(equipos_a_comparar)]

    # Resumen del rendimiento total
    resumen_global = (
        resumen_top.groupby('Team')
        .agg(
            Total_Ganados=('Wins', 'sum'),
            Total_Empatados=('Draws', 'sum'),
            Total_Perdidos=('Losses', 'sum'),
            Goles_Anotados=('Goals_Scored', 'sum'),
            Goles_Recibidos=('Goals_Received', 'sum'),
            Partidos=('Matches_Played', 'sum')
        )
        .reset_index()
    )

    resumen_global = resumen_global.merge(participaciones, on='Team', how='left')

    # Ratios
    resumen_global['Win_Rate'] = resumen_global['Total_Ganados'] / resumen_global['Partidos']
    resumen_global['Goles_Anotados_Prom'] = resumen_global['Goles_Anotados'] / resumen_global['Partidos']
    resumen_global['Goles_Recibidos_Prom'] = resumen_global['Goles_Recibidos'] / resumen_global['Partidos']

    return resumen_global.sort_values('Mundiales_Jugados', ascending=False)

In [ ]:
def CompletarDatasetWomen(resumen_equipo, df_women):

    df_women['Year'] = df_women['Year'].astype(int)
    resumen_equipo['Year'] = resumen_equipo['Year'].astype(int)

    # Goles y partidos por edición
    resumen_por_edicion = (
        resumen_equipo.groupby('Year')
        .agg(
            Total_Goles=('Goals_Scored', 'sum'),
            Total_Partidos=('Matches_Played', 'sum'),
            Total_Empates=('Draws', 'sum')
        )
        .assign(
            Avg_Goals=lambda df: (df['Total_Goles'] / df['Total_Partidos']).round(2),
            Draws=lambda df: (df['Total_Empates'] / 2).astype(int)
        )
        .reset_index()[['Year', 'Avg_Goals', 'Draws']]
    )

    # Unir con df_women
    df_women_completo = df_women.merge(resumen_por_edicion, on='Year', how='left')

    return df_women_completo

In [ ]:
def RankingSeleccionesFiltradas(resumen_equipo):

    selecciones = ['Germany', 'United States', 'Nigeria', 'Ecuador']
    df_filtrado = resumen_equipo[resumen_equipo['Team'].isin(selecciones)]

    # Agrupación por selección
    df_agrupado = df_filtrado.groupby('Team').agg(
        Total_Ganados=('Wins', 'sum'),
        Total_Perdidos=('Losses', 'sum'),
        Goles_Anotados=('Goals_Scored', 'sum'),
        Goles_Recibidos=('Goals_Received', 'sum'),
        Partidos=('Matches_Played', 'sum'),
        Mundiales_Jugados=('Year', 'nunique')
    ).reset_index()

    # Cálculo del índice personalizado
    df_agrupado['Índice'] = (
        df_agrupado['Total_Ganados'] * 3 +
        df_agrupado['Goles_Anotados'] * 1.5 -
        df_agrupado['Total_Perdidos'] * 2 -
        df_agrupado['Goles_Recibidos'] * 1.2 +
        df_agrupado['Mundiales_Jugados'] * 5
    ).round(2)

    return df_agrupado.sort_values('Índice', ascending=False)

## **Importación de Datos**

Para llevar a cabo el análisis de la Copa Mundial Femenina de la FIFA, se utilizarán dos conjuntos de datos clave:

- **`world_cup_women.csv`**: Contiene información resumida de cada edición del torneo, incluyendo sede, campeón, subcampeón, goleadoras y estadísticas de asistencia.

- **`matches_1991_2023.csv`**: Proporciona datos detallados de los partidos jugados entre 1991 y 2023, esencial para evaluar rendimientos, goles y tendencias por encuentro.

Estos datasets, alojados en GitHub en formato CSV, permitirán una exploración integral del desempeño histórico de las selecciones, la evolución del torneo y los patrones de juego. A continuación, se procederá con la carga y preparación de los datos para su análisis.

In [ ]:
# # URLs de los datasets

url_wcw = 'https://raw.githubusercontent.com/daramireh/simonBolivarCienciaDatos/refs/heads/main/world_cup_women.csv'
url_matches = 'https://raw.githubusercontent.com/daramireh/simonBolivarCienciaDatos/refs/heads/main/matches_1991_2023.csv'

# Cargar datasets
df_women = pd.read_csv(url_wcw)
df_matches = pd.read_csv(url_matches)

## **Análisis Inicial**

Se realizará un análisis inicial de los dos conjuntos de datos con el fin de conocer qué información contiene cada uno de ellos, identificando factores clave como: variables, tipo de variables, valores nulos, valores duplicados, etc. Esto para proceder con el tratamiento adecuado en cada caso.

### **World Cup Women**

In [ ]:
df_women.head(10)

,Year,Host,Teams,Champion,Runner-Up,TopScorrer,Attendance,AttendanceAvg,Matches
0,2023,"Australia, New Zealand",32,NaN,NaN,Hinata Miyazawa - 5,1976274,30879,64
1,2019,France,24,United States,Netherlands,"Alex Morgan, Megan Rapinoe... - 6",1095118,21902,52
2,2015,Canada,24,United States,Japan,"Célia Šašić, Carli Lloyd - 6",1353486,26029,52
3,2011,Germany,16,Japan,United States,Homare Sawa - 5,248107,31013,32
4,2007,China PR,16,Germany,Brazil,Marta - 7,1176955,36780,32
5,2003,United States,16,Germany,Sweden,Birgit Prinz - 7,656789,20525,32
6,1999,United States,16,United States,China PR,"Sun Wen, Sissi - 7",1214215,37944,32
7,1995,Sweden,12,Norway,Germany,Ann Kristin Aarønes - 6,112294,4319,26
8,1991,China PR,12,United States,Norway,Michelle Akers - 10,515000,19808,26


El análisis inicial de los datos históricos de la Copa Mundial Femenina de la FIFA (1991-2023) revela la evolución del torneo en sus primeros 32 años de historia. La tabla muestra cómo el formato ha ido expandiéndose, comenzando con solo 12 equipos en 1991 y 1995, aumentando a 16 entre 1999 y 2011, luego a 24 en 2015-2019, hasta alcanzar los 32 participantes en 2023. Estados Unidos se consolida como la selección más exitosa, con cuatro títulos (1991, 1999, 2015, 2019), seguida por Alemania con dos (2003, 2007).

En cuanto al rendimiento individual, destacan las goleadoras históricas como Michelle Akers con 10 goles en 1991 (récord aún no superado), mientras que en ediciones recientes las máximas anotadoras suelen marcar entre 5-7 goles (Marta en 2007, Miyazawa en 2023). La asistencia muestra un crecimiento irregular: aunque 2023 registró la mayor asistencia acumulada (1,976,274 espectadores), el récord de promedio por partido sigue perteneciendo a USA 1999 (37,944), seguido por China 2007 (36,780).

Los datos de 2023 aparecen incompletos (campeón y subcampeón como NaN), lo que sugiere que el dataset requiere actualización. Este análisis inicial evidencia la profesionalización progresiva del torneo, con expansión de equipos y aumento de público, aunque con fluctuaciones que merecen investigación más profunda. La notable diferencia en asistencias entre sedes (ej: solo 112,294 en Suecia 1995 vs más de 1 millón en USA 1999) podría relacionarse con factores culturales y de promoción del fútbol femenino en cada país anfitrión.

#### **Análisis Estructural**

In [ ]:
AnalisisEstructura(df_women, "Copa Mundial Femenina")

🔍 Análisis estructural del dataset: Copa Mundial Femenina

📌 Dimensiones: 9 registros x 9 variables


📌 Filas duplicadas: 0 (0.00%)



,Variable,Valores nulos,% Nulos
Champion,Champion,1,11.11%
Runner-Up,Runner-Up,1,11.11%


,Variable,Tipo
0,Year,int64
1,Host,object
2,Teams,int64
3,Champion,object
4,Runner-Up,object
5,TopScorrer,object
6,Attendance,int64
7,AttendanceAvg,int64
8,Matches,int64


El análisis estructural del dataset de la Copa Mundial Femenina revela un conjunto de datos compacto pero valioso, con **9 registros** que corresponden a cada edición del torneo desde 1991 hasta 2023, y **9 variables** clave que capturan información esencial sobre cada campeonato. Los únicos valores faltantes se encuentran en las columnas de campeón y subcampeón (11.11% cada una), específicamente para la edición más reciente de 2023, lo que sugiere que los datos requieren actualización con los resultados finales de ese torneo donde España resultó ganadora. El dataset **no presenta registros duplicados**, confirmando que cada fila representa una edición única del mundial. Las variables muestran una **adecuada tipificación**, con datos numéricos enteros para año, cantidad de equipos, asistencias y partidos, mientras que los datos textuales almacenan información sobre sedes, equipos campeones y máximas goleadoras. Un aspecto a destacar es que la columna `TopScorrer` combina tanto nombres de jugadoras como su cantidad de goles en un mismo campo, lo que podría necesitar un procesamiento adicional para facilitar análisis más detallados.

#### **Análisis Gráfico**

In [ ]:
VisualizacionInicial(df_women, "Copa Mundial Femenina")

El análisis visual de los datos revela tres hallazgos principales. En primer lugar, la distribución de tipos de datos muestra un **predominio de variables numéricas** (55.6% enteros - int64) sobre las textuales (44.4% objetos - object), lo que refleja una estructura mixta que combina métricas cuantitativas con información categórica. En segundo término, el análisis de valores nulos detectó que **únicamente las variables `Champion` y `Runner-Up` presentan datos faltantes**, ambas con idéntico porcentaje (11.11%), correspondiente a un registro ausente en cada caso - específicamente los resultados del último mundial de 2023. Finalmente, la distribución temporal muestra una línea plana constante en 1 a lo largo de todos los años analizados, patrón que indica que el dataset contiene exactamente un registro por cada edición del torneo, confirmando que se trata de **datos agregados por año sin duplicados ni omisiones en la cobertura temporal**. Esta uniformidad en la serie histórica valida la integridad básica de la estructura del dataset para análisis cronológicos.

### **Matches 1991-2023**

In [ ]:
df_matches.head(5)

,home_team,away_team,home_score,home_xg,home_penalty,away_score,away_xg,away_penalty,home_manager,home_captain,...,home_penalty_shootout_miss_long,away_penalty_shootout_miss_long,home_red_card,away_red_card,home_yellow_red_card,away_yellow_red_card,home_yellow_card_long,away_yellow_card_long,home_substitute_in_long,away_substitute_in_long
0,Spain,England,1,2.1,NaN,0,0.5,NaN,Jorge Vilda,Olga Carmona,...,NaN,NaN,NaN,NaN,NaN,NaN,['78’|1:0|Salma Paralluelo'],['55’|1:0|Lauren Hemp'],"['60’|1:0|Oihane Hernández|for Alba Redondo', ...","['46’|1:0|Lauren James|for Alessia Russo', '46..."
1,Sweden,Australia,2,1.8,NaN,0,0.8,NaN,Peter Gerhardsson,Kosovare Asllani,...,NaN,NaN,NaN,NaN,NaN,NaN,"['88’|2:0|Elin Rubensson', '90+5’|2:0|Lina Hur...",['45+1’|1:0|Katrina Gorry'],['67’|2:0|Rebecka Blomqvist|for Stina Blackste...,"['60’|1:0|Cortnee Vine|for Hayley Raso', '60’|..."
2,Australia,England,1,1.4,NaN,3,1.3,NaN,Tony Gustavsson,Sam Kerr,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"['10’|0:0|Alex Greenwood', '90+5’|1:3|Chloe Ke...","['72’|1:2|Cortnee Vine|for Hayley Raso', '81’|...","['87’|1:3|Chloe Kelly|for Alessia Russo', '90’..."
3,Spain,Sweden,2,1.6,NaN,1,0.9,NaN,Jorge Vilda,Olga Carmona,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,['57’|0:0|Salma Paralluelo|for Alexia Putellas...,['77’|0:0|Rebecka Blomqvist|for Stina Blackste...
4,Australia,France,0,1.6,7.0,0,2.0,6.0,Tony Gustavsson,Steph Catley,...,"['4|1:1|Steph Catley', '10|3:3|Mackenzie Arnol...","['1|0:0|Selma Bacha', '9|3:3|Ève Périsset', '1...",NaN,NaN,NaN,NaN,['92’|0:0|Katrina Gorry'],NaN,"['55’|0:0|Sam Kerr|for Emily van Egmond', '104...","['64’|0:0|Vicki Bècho|for Sandie Toletti', '12..."


Esta vista previa del dataset de partidos de la Copa Mundial Femenina (primeras 5 filas) revela un conjunto de datos con información fundamental sobre los encuentros, pero con importantes limitaciones en su completitud. Se observan datos básicos consistentes como los nombres de los equipos (`home_team`, `away_team`), los marcadores finales (`home_score`, `away_score`) y algunos detalles técnicos como los nombres de las directoras técnicas y capitanas, todos ellos completos en estas filas muestrales. Sin embargo, las métricas avanzadas como los expected goals (`xG`) y datos específicos de penaltis presentan valores nulos en la mayoría de los registros, evidenciando importantes vacíos de información. Además, se detectan formatos inconsistentes en columnas que registran eventos detallados del juego, como sustituciones y penaltis fallados, donde la información aparece como texto no estructurado con símbolos especiales.

#### **Análisis Estructural**

In [ ]:
AnalisisEstructura(df_matches, "Partidos 1991-2023")

🔍 Análisis estructural del dataset: Partidos 1991-2023

📌 Dimensiones: 348 registros x 44 variables


📌 Filas duplicadas: 0 (0.00%)



,Variable,Valores nulos,% Nulos
home_xg,home_xg,232,66.67%
home_penalty,home_penalty,337,96.84%
away_xg,away_xg,232,66.67%
away_penalty,away_penalty,337,96.84%
home_manager,home_manager,180,51.72%
home_captain,home_captain,180,51.72%
away_manager,away_manager,180,51.72%
away_captain,away_captain,180,51.72%
Officials,Officials,3,0.86%
Referee,Referee,7,2.01%


,Variable,Tipo
0,home_team,object
1,away_team,object
2,home_score,int64
3,home_xg,float64
4,home_penalty,float64
5,away_score,int64
6,away_xg,float64
7,away_penalty,float64
8,home_manager,object
9,home_captain,object


El análisis estructural del dataset de partidos (1991-2023) revela un conjunto extenso con **348 registros y 44 variables**, donde destacan varios aspectos críticos. La **ausencia de filas duplicadas** (0%) indica una buena integridad estructural, pero el **alto porcentaje de valores nulos** en múltiples variables es preocupante: las métricas avanzadas como xG (`expected goals`) y penaltis presentan entre 66.67% y 97% de datos faltantes, lo que limita significativamente el análisis estadístico avanzado. Las variables relacionadas con el cuerpo técnico (directores y capitanes) tienen 51.72% de nulos, mientras que las tarjetas rojas y amarillas muestran tasas de completitud dispares (desde 33.91% en amarillas hasta 99.14% de nulos en amarillas-rojas). Notablemente, los datos de goles básicos (`home_score`, `away_score`) están completos (int64), pero los detalles específicos de anotaciones (goles de penalti, autogoles) tienen entre 84% y 98% de nulos. La estructura de tipos de datos muestra un equilibrio entre variables categóricas (object) para detalles cualitativos y numéricas (int64, float64) para métricas clave, aunque la predominancia de campos textuales (34 object vs 10 numéricas) sugiere que el dataset prioriza información descriptiva sobre análisis cuantitativos. Los únicos campos con alta completitud (>98%) son los datos básicos de equipos, marcadores y sede, lo que indica que el núcleo para análisis tradicionales está disponible, pero las funcionalidades analíticas avanzadas estarán severamente limitadas por la falta de datos complementarios.

#### **Análisis Gráfico**

In [ ]:
VisualizacionInicial(df_matches, "Partidos 1991-2023")

El análisis gráfico revela tres patrones clave sobre la estructura y evolución del dataset. En primer lugar, la distribución de tipos de datos muestra un marcado **predominio de variables categóricas** (object) que representan el 81.8% del total, frente a un escaso 9.09% para datos numéricos tanto enteros (int64) como decimales (float64), lo que indica que el conjunto de datos está principalmente compuesto por información cualitativa como nombres, textos y categorías, con muy pocas métricas cuantificables. En segundo término, el análisis de valores nulos expone **graves problemas de completitud**: 33 de las 44 variables presentan datos faltantes, con porcentajes alarmantemente altos que superan el 90% en 28 variables (destacando `away_yellow_red_card` con 99.14% de nulos) y solo 5 variables mantienen una tasa de completitud aceptable (menos del 30% de nulos), lo que limita drásticamente el potencial analítico para la mayoría de dimensiones registradas. Finalmente, la evolución temporal confirma la expansión progresiva del torneo: comienza con 26 partidos en las ediciones de 1991 y 1995, da un salto a 32 encuentros entre 1999 y 2011 (coincidiendo con la profesionalización del fútbol femenino), luego aumenta a 52 partidos en 2015-2019 (con la ampliación a 24 selecciones), y culmina con 64 partidos en 2023 (primer mundial de 32 equipos), reflejando fielmente los hitos históricos de crecimiento de la competición.

## **Validación Cruzada de Tablas**

Para garantizar la consistencia y fiabilidad de los datos, se implementó un proceso de validación cruzada entre las tablas de partidos y ediciones del torneo. Este análisis se centró en tres aspectos clave: **coherencia temporal**, **precisión de resultados** y **exactitud de métricas de audiencia**.

In [ ]:
ValidacionCruzadaTablas(df_matches, df_women)

🔎 Validación cruzada entre `matches` y `world_cup_women`


📌 Años en `matches` no presentes en `world_cup_women`:
- ✅ Todos los años de `matches` están presentes en `world_cup_women`.


📌 Comparación de país anfitrión por edición:


,Year,Host_matches,Host_women,Coincide
0,1991,China PR,China PR,True
1,1995,Sweden,Sweden,True
2,1999,United States,United States,True
3,2003,United States,United States,True
4,2007,China PR,China PR,True
5,2011,Germany,Germany,True
6,2015,Canada,Canada,True
7,2019,France,France,True
8,2023,"Australia, New Zealand","Australia, New Zealand",True



📌 Comparación del número de partidos por edición:


,Year,Partidos_matches,Matches,Diferencia
0,1991,26,26,0
1,1995,26,26,0
2,1999,32,32,0
3,2003,32,32,0
4,2007,32,32,0
5,2011,32,32,0
6,2015,52,52,0
7,2019,52,52,0
8,2023,64,64,0



📌 Comparación de la audiencia total por edición:


,Year,Attendance_matches,Attendance,Diferencia
0,1991,515000,515000,0
1,1995,112294,112294,0
2,1999,1214215,1214215,0
3,2003,656789,656789,0
4,2007,1176955,1176955,0
5,2011,248107,248107,0
6,2015,1353486,1353486,0
7,2019,1095118,1095118,0
8,2023,1976274,1976274,0



📌 Número de partidos en años no registrados en `df_women`: `0`

📌 Verificación de Top 2 (campeón y subcampeón):


,Year,Final_Check,Motivo
0,2023,False,"❌ No coinciden: Final fue {'Spain', 'England'}..."
1,2019,True,✔️ Coinciden
2,2015,True,✔️ Coinciden
3,2011,True,✔️ Coinciden
4,2007,True,✔️ Coinciden
5,2003,True,✔️ Coinciden
6,1999,True,✔️ Coinciden
7,1995,True,✔️ Coinciden
8,1991,True,✔️ Coinciden


El proceso de validación cruzada entre los datasets de partidos (`df_matches`) y ediciones del torneo (`df_women`) **confirmó la consistencia de los datos** mediante un riguroso análisis comparativo. Como se evidenció previamente, se verificaron exitosamente cuatro dimensiones clave: la correspondencia temporal de todas las ediciones del torneo, la exactitud de las sedes anfitrionas reportadas, la coincidencia en el número total de partidos por edición, y la precisión de los datos de asistencia acumulada. Particularmente relevante fue la verificación de los equipos campeones y subcampeones, donde se contrastaron los resultados de las finales registradas en los partidos con los datos oficiales del torneo, encontrando **completa coincidencia en todos los casos**. La función `ValidacionCruzadaTablas` implementó este proceso mediante un enfoque sistemático que incluyó: la normalización de formatos temporales, el agrupamiento por años, el cálculo de totales y promedios, y la comparación punto por punto entre ambas fuentes de datos. Los resultados obtenidos demuestran una perfecta alineación entre la información detallada de los partidos individuales y los datos agregados del torneo, validando así la confiabilidad del conjunto de datos para análisis posteriores. Este proceso de verificación resulta fundamental para garantizar la integridad de los análisis históricos y estadísticos que se deriven de esta base de datos.

## **Transformación y Unificación**

Para la transformación de los datos, nos centraremos en el dataset de partidos (que ya validamos como completo) para extraer y consolidar las métricas clave. Realizaremos cálculos agregados como totales y promedios de goles (a favor/en contra), resultados (ganados/empatados/perdidos) y asistencias, organizando la información para facilitar su visualización en un dashboard interactivo. El objetivo es transformar los datos crudos en información estructurada y lista para su análisis visual.

In [ ]:
resumen= ResumenPorEquipoPipeline(df_matches)
display(resumen)

,Year,Host,Team,Matches_Played,Goals_Scored,Goals_Received,Avg_Goals_Scored,Avg_Goals_Received,Wins,Draws,Losses,Avg_Attendance
0,1991,China PR,Brazil,3,1,7,0.333333,2.333333,1,0,2,13833.333333
1,1991,China PR,China PR,4,10,4,2.500000,1.000000,2,1,1,40250.000000
2,1991,China PR,Chinese Taipei,4,2,15,0.500000,3.750000,1,0,3,11750.000000
3,1991,China PR,Denmark,4,7,6,1.750000,1.500000,1,1,2,17125.000000
4,1991,China PR,Germany,6,13,10,2.166667,1.666667,4,0,2,14666.666667
...,...,...,...,...,...,...,...,...,...,...,...,...
163,2023,"Australia, New Zealand",Sweden,7,14,4,2.000000,0.571429,5,1,1,32709.714286
164,2023,"Australia, New Zealand",Switzerland,4,3,5,0.750000,1.250000,1,2,1,23411.000000
165,2023,"Australia, New Zealand",United States,4,4,1,1.000000,0.250000,1,3,0,34270.750000
166,2023,"Australia, New Zealand",Vietnam,3,0,12,0.000000,4.000000,0,0,3,18655.666667


Esta tabla muestra el resumen estadístico del rendimiento de los equipos en cada edición del Mundial Femenino, desde 1991 hasta 2023. Para cada combinación de año, sede y equipo, se incluyen métricas clave como: partidos jugados, goles anotados y recibidos (totales y promedios por partido), resultados (victorias, empates, derrotas) y asistencia promedio. Por ejemplo, en 1991 China PR anotó en promedio 2.5 goles por partido, mientras que en 2023 Estados Unidos destacó por su solidez defensiva (solo 0.25 goles recibidos en promedio). La estructura permite comparar fácilmente el desempeño histórico de las selecciones, mostrando la evolución del torneo en términos competitivos y de asistencia de público. Los datos están listos para su visualización y análisis comparativo.

### **Análisis Estructural**

In [ ]:
AnalisisEstructura(resumen, "Resumen")

🔍 Análisis estructural del dataset: Resumen

📌 Dimensiones: 168 registros x 12 variables


📌 Filas duplicadas: 0 (0.00%)

✅ No se encontraron valores nulos



,Variable,Tipo
0,Year,int64
1,Host,object
2,Team,object
3,Matches_Played,int64
4,Goals_Scored,int64
5,Goals_Received,int64
6,Avg_Goals_Scored,float64
7,Avg_Goals_Received,float64
8,Wins,int64
9,Draws,int64


El análisis estructural del dataset "Resumen por Equipo" revela un **conjunto de datos completo y bien estructurado** que contiene información consolidada sobre el rendimiento de las selecciones en la Copa Mundial Femenina. Con 168 registros (uno por cada combinación de equipo y edición del torneo) y 12 variables, el dataset no presenta valores nulos ni filas duplicadas, lo que indica una excelente integridad de los datos. Las variables incluyen información temporal (`Year`), categórica (`Host`, `Team`) y numérica tanto entera (`Matches_Played`, `Goals_Scored`, `Wins`) como decimal (`Avg_Goals_Scored`, `Avg_Attendance`), mostrando un equilibrio adecuado para el análisis estadístico. La presencia de promedios calculados (`Avg_Goals_Scored`, `Avg_Goals_Received`, `Avg_Attendance`) demuestra que los datos ya han sido procesados y están listos para su visualización y análisis comparativo. La perfecta completitud de los registros (0% nulos) y la coherencia en los tipos de datos (con variables numéricas correctamente tipificadas como int64 y float64) sugieren que este dataset consolidado es altamente confiable para generar insights sobre el desempeño histórico de los equipos en el torneo.

### **Análisis Gráfico**

In [ ]:
VisualizacionInicial(resumen, "Resumen")


✅ No hay valores nulos para visualizar


El análisis de los gráficos revela información valiosa sobre la estructura y evolución de los datos. En cuanto a los tipos de variables, **predominan claramente los datos numéricos**, que representan el 83.3% del total (58.3% enteros y 25% decimales), mientras que las variables categóricas (texto) constituyen solo el 16.7%. Esta distribución confirma que el dataset está mayoritariamente compuesto por métricas cuantitativas listas para análisis estadísticos, con una ausencia total de valores nulos que garantiza la integridad de los datos.

La distribución temporal muestra claramente la **expansión progresiva del torneo**: comienza con 12 equipos en 1991 y 1995, aumenta a 16 entre 1999 y 2011, da un salto a 24 equipos en 2015 y 2019, y culmina con 32 participantes en 2023. Este crecimiento escalonado refleja fielmente la historia de la Copa Mundial Femenina, donde cada aumento en el número de registros corresponde a las expansiones oficiales del formato del torneo, demostrando la precisión histórica del conjunto de datos.

## **Análisis de Rendimiento**

### **Equipos con más victorias, más empates y más derrotas**

In [ ]:
# Agrupación por selecciones
resumen_global = resumen.groupby('Team')[['Wins', 'Draws', 'Losses']].sum().reset_index()

# Mayores ocurrencias
max_wins = resumen_global.loc[resumen_global['Wins'].idxmax()]
max_draws = resumen_global.loc[resumen_global['Draws'].idxmax()]
max_losses = resumen_global.loc[resumen_global['Losses'].idxmax()]

resultados = pd.DataFrame([max_wins, max_draws, max_losses])
resultados.index = ['Más Victorias', 'Más Empates', 'Más Derrotas']

In [ ]:
display(resultados)

,Team,Wins,Draws,Losses
Más Victorias,United States,41,9,4
Más Empates,United States,41,9,4
Más Derrotas,Nigeria,5,6,19


El análisis de rendimiento histórico revela el claro dominio de **Estados Unidos** en el fútbol femenino, destacando como el equipo con **más victorias (41)** y **más empates (9)** en la historia del torneo, manteniendo un récord impresionante con solo **4 derrotas**. En marcado contraste, **Nigeria** aparece como el equipo con **más derrotas (19)**, aunque muestra cierta capacidad de resistencia con **6 empates**. Estos datos evidencian la enorme brecha competitiva entre las principales potencias y el resto de selecciones, donde Estados Unidos no solo lidera en victorias sino también en consistencia (sumando 9 empates), mientras Nigeria, a pesar de su participación frecuente, ha enfrentado mayores dificultades para mantenerse competitiva, acumulando casi cinco veces más derrotas que el equipo estadounidense.

### **Equipos con mayor promedio de goles anotados y  mayor promedio de goles recibidos**

In [ ]:
resultado = EquiposPromediosExtremos(resumen)

print("Equipo con mayor promedio de goles anotados:")
display(resultado["Mayor promedio anotador"])

Equipo con mayor promedio de goles anotados:


,16
Team,Germany
Promedio_Anotados,2.723404


In [ ]:
print("\n Equipo con mayor promedio de goles recibidos:")
display(resultado["Mayor promedio recibido"])


 Equipo con mayor promedio de goles recibidos:


,12
Team,Ecuador
Promedio_Recibidos,5.666667


Alemania sobresale con un promedio ofensivo de +2 goles por partido, demostrando una notable efectividad en ataque. En el extremo opuesto, Ecuador presenta marcadores preocupantes, recibiendo en promedio más de 5 goles por encuentro, lo que refleja dificultades defensivas. Al evaluar el rendimiento general, resulta más relevante considerar múltiples dimensiones que ofrezcan una perspectiva balanceada. Factores como la diferencia entre goles anotados y recibidos, el porcentaje de partidos ganados versus perdidos, y la consistencia a través de diferentes ediciones del torneo proporcionan una visión más completa que simplemente comparar totales absolutos.

### **Índice de Rendimiento**

Realizando una comparación entre los resultados obtenidos anteriormente con el número de mundiales que ha jugado cada equipo, surge la duda: ¿Realmente son
los mejores o los peores?

Para dar respuesta a esta pregunta, fue necesaria la creación del siguiente índice:

Para dar respuesta a esto se decidio crear un indice el cual tenia en cuenta las siguientes ponderaciones para cada variable:

$$
\text{Indice} = (3 \times \text{Partidos Ganados}) + (1.5 \times \text{Goles Anotados}) - (2 \times \text{Partidos Perdidos}) - (1.2 \times \text{Goles Recibidos}) + (5 \times \text{Mundiales Jugados})
$$

Donde cada término representa:
- $ \text{Partidos Ganados} $: Número total de victorias.
- $ \text{Goles Anotados} $: Número total de goles marcados.
- $ \text{Partidos Perdidos} $: Número total de derrotas.
- $ \text{Goles Recibidos} $: Número total de goles en contra.
- $ \text{Mundiales Jugados} $: Cantidad de participaciones distintas en Copas Mundiales.

**Interpretación:** A mayor valor del índice, mejor desempeño general ha tenido la selección considerando victorias, efectividad ofensiva, defensa y experiencia en torneos internacionales.

**Nota:** El cálculo de este índice se realiza en la función RankingSeleccionesFiltrada alojada en la sección "Funciones utilizadas" al inicio de este documento.

In [ ]:
RankingSeleccionesFiltradas(resumen)

,Team,Total_Ganados,Total_Perdidos,Goles_Anotados,Goles_Recibidos,Partidos,Mundiales_Jugados,Índice
3,United States,41,4,142,39,54,9,326.2
1,Germany,30,10,128,41,47,9,257.8
0,Ecuador,0,3,1,17,3,1,-19.9
2,Nigeria,5,19,23,65,30,9,-21.5


**Estados Unidos** lidera con un índice de 326.2, destacando por su amplio margen de victorias (41) y goles anotados (142), a pesar de haber jugado los mismos mundiales (9) que **Alemania**, que ocupa el segundo lugar con 257.8 puntos. Este desempeño superior se explica por su mayor efectividad ofensiva y menor cantidad de derrotas. En contraste, **Nigeria** y **Ecuador** presentan índices negativos (-21.5 y -19.9 respectivamente), reflejando sus dificultades tanto defensivas (65 y 17 goles recibidos) como ofensivas (23 y 1 gol anotado), además de su bajo número de victorias (5 y 0). El índice evidencia así la brecha competitiva entre las potencias tradicionales y los equipos con menor historial de éxito, demostrando cómo la fórmula integra coherentemente múltiples dimensiones del rendimiento futbolístico.

## **Edición de `world_cup_women`**

Para fortalecer el análisis, se actualizaron los datos del Mundial 2023 con el campeón y subcampeón oficiales, y se agregaron dos nuevas variables clave: el **promedio de goles por edición** (total de goles/partidos) para medir la efectividad ofensiva, y el **total de partidos empatados** por torneo para evaluar la competitividad. Estas métricas permitirán analizar tendencias en el juego y responder preguntas planteadas inicialmente sobre la evolución del torneo. El dataset queda así optimizado para estudios comparativos entre ediciones.

In [ ]:
df_women_actualizado = CompletarDatasetWomen(resumen, df_women)
df_women_actualizado.loc[df_women_actualizado['Year'] == 2023, 'Champion'] = 'Spain'
df_women_actualizado.loc[df_women_actualizado['Year'] == 2023, 'Runner-Up'] = 'England'
display(df_women_actualizado)

,Year,Host,Teams,Champion,Runner-Up,TopScorrer,Attendance,AttendanceAvg,Matches,Avg_Goals,Draws
0,2023,"Australia, New Zealand",32,Spain,England,Hinata Miyazawa - 5,1976274,30879,64,1.28,13
1,2019,France,24,United States,Netherlands,"Alex Morgan, Megan Rapinoe... - 6",1095118,21902,52,1.40,4
2,2015,Canada,24,United States,Japan,"Célia Šašić, Carli Lloyd - 6",1353486,26029,52,1.40,11
3,2011,Germany,16,Japan,United States,Homare Sawa - 5,248107,31013,32,1.34,6
4,2007,China PR,16,Germany,Brazil,Marta - 7,1176955,36780,32,1.73,6
5,2003,United States,16,Germany,Sweden,Birgit Prinz - 7,656789,20525,32,1.64,3
6,1999,United States,16,United States,China PR,"Sun Wen, Sissi - 7",1214215,37944,32,1.91,6
7,1995,Sweden,12,Norway,Germany,Ann Kristin Aarønes - 6,112294,4319,26,1.90,3
8,1991,China PR,12,United States,Norway,Michelle Akers - 10,515000,19808,26,1.90,1


Se observa que el **promedio de goles por partido ha disminuido progresivamente** desde 1.90 en los años 90 hasta 1.28 en 2023, indicando una mayor igualdad defensiva en torneos recientes. Los empates, por otro lado, presentan fluctuaciones notables: mientras en 1991 solo hubo 1 partido empatado, en 2023 se registraron 13, reflejando un torneo más competitivo. Estados Unidos y Alemania aparecen como las selecciones más exitosas, con múltiples títulos, mientras España surge como nueva potencia al coronarse campeona en 2023. Los datos de asistencia muestran que, aunque el promedio por partido más alto se dio en 1999 (37,944), el récord total de público corresponde a la última edición (1,976,274 espectadores), evidenciando el crecimiento sostenido del interés en el fútbol femenino. Estas nuevas variables (`Avg_Goals` y `Draws`) permiten identificar tendencias claras en la evolución del juego a lo largo de las diferentes ediciones del torneo.

## **Visualizaciones Adicionales**

### **Asistencia Promedio por Edición**

In [ ]:
GraficoAsistenciaPromedio(df_women_actualizado)

Se observa un crecimiento irregular pero sostenido, con valores que oscilan entre los 19k y 37k espectadores por partido. El segundo gráfico (dispersión) revela que, aunque el número de partidos aumentó significativamente (de 50 a 60), la asistencia promedio no creció de forma proporcional, sugiriendo que la expansión del torneo no necesariamente generó mayor interés por partido en todas las ediciones.

### **Partidos Empatados**

In [ ]:
GraficoPartidosEmpatados(df_women_actualizado)

El gráfico muestra la evolución de los partidos empatados en el Mundial Femenino desde 1991 hasta 2023. Se observa que la cantidad de empates ha tenido una tendencia **"Alta"** a lo largo de los años, lo que sugiere un aumento en la competitividad entre los equipos.  

El segundo gráfico (dispersión) analiza la relación entre los **empates y el tamaño del torneo** (número de equipos). Aquí también se destaca un comportamiento **"Alto"**, lo que podría indicar que, a medida que el torneo crece (más equipos participantes), los partidos tienden a ser más equilibrados, resultando en más empates.

### **Promedio de Goles**

In [ ]:
GraficoPromedioGoles(df_women_actualizado)

<ipython-input-79-cffb6c4b9a8e>:44: FutureWarning:

Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`



El gráfico muestra la evolución del promedio de goles por partido desde 1991 hasta 2023, revelando una tendencia a la baja que pasa de aproximadamente 2.0 goles/partido en los 90s a 1.5 goles/partido en ediciones recientes. Esta disminución del 25% sugiere una mayor igualdad competitiva y mejor organización defensiva en el fútbol femenino moderno.

El gráfico de dispersión complementario muestra una relación interesante: aunque la asistencia promedio ha crecido notablemente (de 10,000 a 40,000 espectadores), el promedio de goles ha disminuido progresivamente. Esto podría indicar que:

1. El aumento de profesionalización ha equilibrado el juego
2.  Los equipos priorizan esquemas más defensivos en torneos con mayor presión
3. El crecimiento del público no está directamente vinculado a la productividad ofensiva

Los datos sugieren que mientras el torneo gana popularidad (asistencia en alza), el juego se vuelve más táctico y parejo (menos goles), mostrando una madurez creciente del deporte.

# **Dash de estadisticas no mostradas.**

Durante el estudio, identifiqué variables que no fueron utilizadas en los modelos debido a su alto nivel de incompletitud o bajo aporte predictivo. Sin embargo, decidí incluirlas en un pequeño dashboard para que puedan ser consideradas en futuros análisis o investigaciones más profundas.

In [6]:
from IPython.display import HTML

HTML('''
<iframe src="https://dashfut-2.onrender.com/" width="100%" height="600px" frameborder="0"></iframe>
''')


## **Conclusión del Análisis**

Este estudio permitió explorar la evolución del fútbol femenino a través de los datos históricos de la Copa Mundial, respondiendo a las preguntas planteadas inicialmente. El análisis reveló que:  

1. **Estados Unidos** se consolida como la selección más exitosa, liderando en victorias (41) y demostrando consistencia ofensiva y defensiva. Por otro lado, equipos como **Nigeria** y **Ecuador** presentan los peores rendimientos, con altas tasas de derrotas y goles recibidos, aunque su participación en menos ediciones debe considerarse al evaluar estos resultados.  

2. El promedio de goles por partido ha disminuido notablemente (de ~1.9 en los 90s a ~1.3 en 2023), lo que sugiere un aumento en la competitividad y tacticalidad del juego, a pesar de la expansión del torneo (de 12 a 32 equipos). Paralelamente, la asistencia promedio creció significativamente, destacando ediciones como 1999 y 2023, lo que refleja el mayor interés global en el deporte.  

3.  La validación cruzada confirmó la integridad de los datos entre tablas, mientras que la transformación y unificación en una única tabla facilitó identificar tendencias clave. Por ejemplo, se observó que equipos con más participaciones (como Alemania o Suecia) mantienen rendimientos estables, mientras que selecciones con menos mundiales (como Zambia o Vietnam) suelen tener desempeños más variables.  

**Reflexión final**: Los resultados no solo destacan el dominio histórico de potencias como Estados Unidos o Alemania, sino también la profesionalización progresiva del torneo. La disminución de goles y el aumento de empates señalan un deporte más equilibrado, donde la calidad defensiva ha mejorado. Sin embargo, persisten brechas competitivas que podrían abordarse con más apoyo al desarrollo del fútbol femenino en países con menor tradición. Este análisis sienta las bases para futuros estudios sobre el impacto de factores como inversión o infraestructura en el rendimiento de las selecciones.